# Hebrew OCR — try it in Colab

[`paddleocr-hebrew`](https://github.com/RivoksLab/paddleocr-hebrew) — production Hebrew OCR,
finetuned from PaddleOCR and released Apache-2.0 by [Rivok Labs](https://rivoklabs.com).

Runs on a **free CPU runtime** — no GPU needed, nothing to install locally.
Runtime → *Run all*, then upload your own page in the last cell.

Strong on the two things generic OCR gets wrong for Hebrew: real-world documents
(menus, receipts, invoices, scans) and **bilingual Hebrew + Latin/digit lines**.

[Models](https://huggingface.co/Rivok/paddleocr-hebrew) ·
[Code](https://github.com/RivoksLab/paddleocr-hebrew) ·
[How it works](https://rivoklabs.com/research/hebrew-ocr.html)


## 1. Install

About a minute. No runtime restart needed.


In [ ]:
!pip install -q git+https://github.com/RivoksLab/paddleocr-hebrew.git huggingface_hub


## 2. Load the models

Pulls ~180 MB from the Hugging Face Hub on first run, then caches it for the session.
Only the flagship path is fetched; the repo also holds mobile and alternate recognizers.


In [ ]:
from huggingface_hub import snapshot_download
from paddleocr_hebrew import HebrewOCR

models_dir = snapshot_download(
    repo_id="Rivok/paddleocr-hebrew",
    allow_patterns=["charset_v2f.txt", "word-det/*", "line-det/*", "server-svtrv2/*"],
)

# .word() is the flagship. .line() is for degraded scans or dense layouts
# where word boxes over-fragment.
ocr = HebrewOCR.word(models_dir)
print("ready")


## 3. Read a sample page

Note the line with `University of Haifa 2024`, the catalogue number and the decimals.
Those embedded left-to-right runs are what a plain CTC recognizer tends to delete
outright; the script-gated cascade switches to an attention decoder on exactly
those crops.


In [ ]:
!wget -q https://raw.githubusercontent.com/RivoksLab/paddleocr-hebrew/main/examples/sample_images/sample_page.png

from IPython.display import Image, display
display(Image("sample_page.png"))


In [ ]:
result = ocr.read("sample_page.png")

meta = result["meta"]
print(f"{meta['n_lines']} lines · {meta['n_words']} words · "
      f"{meta['n_nrtr_fallback']} attention fallbacks\n")

for line in result["lines"]:
    print(line["text"])


## 4. Your own page

Run the cell, pick a PNG, JPG, TIFF or PDF, and it reads the first page.


In [ ]:
from google.colab import files

uploaded = files.upload()
path = next(iter(uploaded))

result = ocr.read(path)          # add page=2 for a later PDF page
for line in result["lines"]:
    print(line["text"])


## A note on RTL, worth 30 seconds

Everything above is **logical Unicode order** — the order you read the characters.
That is the correct form to store, search, score and train on, and it is what the
browser renders correctly here.

`python-bidi`'s `get_display()` rearranges text into *visual* order for renderers
that cannot do BiDi themselves, like a bare terminal. Call it at the last step before
a human eye and **never** before storage, training or CER scoring — apply it early and
you silently corrupt your text while every accuracy number you compute afterwards still
looks fine. That mistake cost us 114,755 corrupted annotations.

```python
from bidi.algorithm import get_display
print(get_display(line["text"]))   # display ONLY
```

**Known limitation:** reading-order assembly uses paragraph-direction sorting rather
than full UAX#9 BiDi, so a Hebrew-dominant line with a long embedded Latin run can
order the runs by position rather than strict logical BiDi.

---

If it works well on your documents, or fails on them, we would like to hear about it —
[open an issue](https://github.com/RivoksLab/paddleocr-hebrew/issues) or email
ronen@rivoklabs.com.
